# Check files status (IVDRS & SUDORS)

Compares the workbook's **Files** tab (every file we expect) with what is actually in your `files` folder,
whichever notebook (or you, by hand) saved it. Run it any time; it only reads, it changes nothing.

It reports:
- **done**: the file is there and opens as a real PDF
- **still needed**: missing, empty, not really a PDF (e.g. a web page saved with .pdf), or unreadable,
  with the link and what to do
- **to check**: files that look suspicious (two records with the identical file, very small files,
  Infogram captures with no extra view pages)
- **extra files** in `files/` that aren't on the Files tab (PNG backups, partial downloads, not_open_access/)
  so you can clear them out before the Prism upload

Needs only `openpyxl` and `pypdf` (already installed for the other notebooks).

## Cell 1: Settings ✏️

In [1]:
from pathlib import Path

WORKBOOK  = Path("2026_10-22 IVDRS_SUDORS_Reconciled_Deposit_List.xlsx")
SHEET     = "Files"            # or "Not Open Access Files" to check the review-only articles
FILES_DIR = Path("files")
STATUS_CSV = Path("files_status.csv")   # full results are saved here

for p in (WORKBOOK, FILES_DIR):
    print(f"{'found  ' if p.exists() else 'MISSING'}  {p.resolve()}")

found    C:\Users\keg827\Documents\Github_Repos\Galter_Prism_IVDRS_SUDORS_Post_Request\2026_10-22 IVDRS_SUDORS_Reconciled_Deposit_List.xlsx
found    C:\Users\keg827\Documents\Github_Repos\Galter_Prism_IVDRS_SUDORS_Post_Request\files


## Cell 2: Scan the folder against the Files tab

In [2]:
import hashlib, openpyxl
from collections import defaultdict
from pypdf import PdfReader

ws = openpyxl.load_workbook(WORKBOOK, data_only=True)[SHEET]
head = [c.value for c in ws[1]]
expected = [dict(zip(head, r)) for r in ws.iter_rows(min_row=2, values_only=True)
            if r[0] and str(r[0]).startswith("IVSU")]

def check(path):
    """Returns (state, pages, size, sha256, note)."""
    if not path.exists():
        return "missing", 0, 0, "", "not in the folder"
    size = path.stat().st_size
    if size == 0:
        return "bad", 0, 0, "", "file is empty (0 bytes)"
    data = path.read_bytes()
    sha = hashlib.sha256(data).hexdigest()
    if not data[:5] == b"%PDF-":
        kind = "a web page (HTML)" if b"<html" in data[:2000].lower() else "not a PDF"
        return "bad", 0, size, sha, f"saved file is {kind}, not a real PDF"
    try:
        pages = len(PdfReader(str(path)).pages)
    except Exception as e:
        return "bad", 0, size, sha, f"PDF won't open: {e.__class__.__name__}"
    return "done", pages, size, sha, ""

results = []
for r in expected:
    f = FILES_DIR / r["file_name"]
    state, pages, size, sha, note = check(f)
    results.append({**{k: r.get(k) for k in ("record_id", "file_order", "file_name", "Title", "Link type",
                                             "Download from", "DOI link", "What to do")},
                    "state": state, "pages": pages, "bytes": size, "sha256": sha, "note": note})

# Things worth a second look
by_hash = defaultdict(list)
for x in results:
    if x["sha256"]:
        by_hash[x["sha256"]].append(x)
for group in by_hash.values():
    if len({g["record_id"] for g in group}) > 1:
        for g in group:
            others = ", ".join(sorted({o["record_id"] for o in group} - {g["record_id"]}))
            g["note"] = (g["note"] + "; " if g["note"] else "") + f"identical file to {others}: check both are the right document"
            g["state"] = "check" if g["state"] == "done" else g["state"]
def flag(x, note):
    x["state"] = "check"
    x["note"] = (x["note"] + "; " if x["note"] else "") + note
for x in results:
    if x["state"] in ("done", "check") and x["bytes"] < 20_000:
        flag(x, "very small file (under 20 KB): may be an error page or a cover sheet")
    if x["state"] in ("done", "check") and x["Link type"] == "Infogram web visualization" and x["pages"] == 1:
        flag(x, "Infogram capture has no extra view pages: fine if the report has no tabs/dropdowns")

# Files in the folder that aren't on the Files tab
listed = {(FILES_DIR / x["file_name"]).resolve() for x in results}
extras = sorted(p for p in FILES_DIR.rglob("*") if p.is_file() and p.resolve() not in listed) if FILES_DIR.exists() else []
# For each missing file, look for a near-match: any file in the folder that starts with the same record_id
def explain(found, expected_rel):
    # Say how a near-match differs from the expected path.
    rel = found.relative_to(FILES_DIR)
    exp = Path(expected_rel)
    why = []
    if rel.parent != exp.parent:
        why.append(f"it is in '{rel.parent}' but should be in '{exp.parent}'")
    if rel.name != exp.name:
        if rel.name.lower() == exp.name.lower():
            why.append("only the capital letters differ")
        elif rel.name.lower().endswith(".pdf.pdf"):
            why.append("it ends in '.pdf.pdf' (Windows hides the first .pdf)")
        else:
            why.append(f"the name differs: '{rel.name}'")
    return f"{rel}  ->  " + ("; ".join(why) or "same path?")

extra_by_rid = defaultdict(list)
for p in extras:
    rid = p.name[:9]
    if rid.startswith("IVSU-"):
        extra_by_rid[rid].append(p)
for x in results:
    if x["state"] == "missing" and extra_by_rid.get(x["record_id"]):
        x["near"] = [explain(p, x["file_name"]) for p in extra_by_rid[x["record_id"]]]
        x["note"] = "not at the expected path, but a similar file is in the folder (see below)"

print(f"Checked {len(results)} expected files for {len({x['record_id'] for x in results})} records; "
      f"{len(extras)} other file(s) in {FILES_DIR}/")

Checked 116 expected files for 86 records; 26 other file(s) in files/


## Cell 3: Summary

In [3]:
from collections import Counter

states = Counter(x["state"] for x in results)
print("Files:", dict(states))

recs = defaultdict(list)
for x in results:
    recs[x["record_id"]].append(x["state"])
complete = [r for r, s in recs.items() if all(v in ("done", "check") for v in s)]
print(f"Records with every file present: {len(complete)} of {len(recs)}")

print("\nBy folder:")
folders = defaultdict(Counter)
for x in results:
    folders[str(Path(x["file_name"]).parent)][x["state"]] += 1
for f, c in sorted(folders.items()):
    print(f"  {f:<55} " + ", ".join(f"{k}: {v}" for k, v in sorted(c.items())))

Files: {'done': 113, 'check': 3}
Records with every file present: 86 of 86

By folder:
  articles                                                done: 39
  ivdrs_reports                                           check: 2, done: 33
  ivdrs_reports\ivdrs_county_reports_2016_data            done: 16
  sudors_reports                                          check: 1, done: 9
  sudors_reports\sudors_county_reports_2018               done: 16


## Cell 4: Still needed

In [4]:
need = [x for x in results if x["state"] in ("missing", "bad")]
if not need:
    print("Nothing missing: every file on the Files tab is in the folder and opens as a PDF.")
for x in need:
    print(f"{x['record_id']}  [{x['state'].upper()}: {x['note']}]")
    print(f"    save as:  {FILES_DIR / x['file_name']}")
    print(f"    title:    {x['Title']}")
    print(f"    get from: {x['Download from'] or x['DOI link'] or '(no link)'}")
    if x["DOI link"] and x["DOI link"] != x["Download from"]:
        print(f"    DOI:      {x['DOI link']}")
    print(f"    how:      {x['What to do']}")
    for n in x.get("near", []):
        print(f"    FOUND SIMILAR FILE: {n}")
        print(f"        fix: rename/move it to exactly  {x['file_name']}")

Nothing missing: every file on the Files tab is in the folder and opens as a PDF.


## Cell 5: To check

In [5]:
chk = [x for x in results if x["state"] == "check"]
if not chk:
    print("Nothing flagged.")
for x in chk:
    print(f"{x['record_id']}  {x['file_name']}  ({x['pages']} page(s), {x['bytes']:,} bytes)\n    {x['note']}")

IVSU-0002  ivdrs_reports/IVSU-0002_Veteran_Suicide_Deaths_in_Illinois_2019_2023.pdf  (1 page(s), 210,041 bytes)
    Infogram capture has no extra view pages: fine if the report has no tabs/dropdowns
IVSU-0009  ivdrs_reports/IVSU-0009_Analysis_of_Homicide_Suicide_Incidents_in_Illinois_2015_2020.pdf  (1 page(s), 335,020 bytes)
    Infogram capture has no extra view pages: fine if the report has no tabs/dropdowns
IVSU-0040  sudors_reports/IVSU-0040_Drug_Overdose_Deaths_with_Carfentanil_as_a_Cause_of_Death.pdf  (1 page(s), 283,643 bytes)
    Infogram capture has no extra view pages: fine if the report has no tabs/dropdowns


## Cell 6: Extra files in the folder

Not on the Files tab, so they won't be uploaded. Move or delete them before the Prism upload
(PNG backups from the capture notebook, `.part` leftovers from interrupted downloads, `not_open_access/`).

In [6]:
if not extras:
    print("None.")
groups = defaultdict(list)
for p in extras:
    rel = p.relative_to(FILES_DIR)
    key = "not_open_access/ (review-only articles)" if rel.parts[0] == "not_open_access" else \
          "PNG backups from the capture notebook" if p.suffix.lower() == ".png" else \
          "partial downloads (.part / .tmp.pdf)" if p.suffix.lower() == ".part" or p.name.endswith(".tmp.pdf") else \
          "other"
    groups[key].append(rel)
for k, v in groups.items():
    print(f"{k}: {len(v)}")
    for rel in v[:50]:
        print(f"    {rel}")
    if len(v) > 50:
        print(f"    ... and {len(v) - 50} more")

PNG backups from the capture notebook: 15
    ivdrs_reports\IVSU-0001_Child_Abuse_Fatalities_in_Illinois_2015_2022.png
    ivdrs_reports\IVSU-0002_Veteran_Suicide_Deaths_in_Illinois_2019_2023.png
    ivdrs_reports\IVSU-0003_Intimate_Partner_Problem_Related_Suicide_and_Homicide_in_Ill.png
    ivdrs_reports\IVSU-0006_Analysis_of_Older_Adult_Suicide_Deaths_in_Illinois_2020_2021.png
    ivdrs_reports\IVSU-0007_Child_and_Young_Adult_Suicide_Focus_on_Incidents_with_an_Int.png
    ivdrs_reports\IVSU-0008_Illinois_Veteran_Suicide_Deaths_2021.png
    ivdrs_reports\IVSU-0009_Analysis_of_Homicide_Suicide_Incidents_in_Illinois_2015_2020.png
    ivdrs_reports\IVSU-0010_Illinois_Homicide_Deaths_2020.png
    ivdrs_reports\IVSU-0125_Suicide_Deaths_Among_Females_in_Illinois_2018_2023.png
    ivdrs_reports\IVSU-0126_Intimate_Partner_Problem_related_Violence_Deaths_in_Illinois.png
    sudors_reports\IVSU-0036_Overdose_Deaths_Among_Females_in_Illinois_2019_2024.png
    sudors_reports\IVSU-0037_Veteran_Ove

## Cell 7: Save the full results

In [7]:
import csv
cols = ["record_id", "file_order", "file_name", "state", "pages", "bytes", "note", "Title", "Link type",
        "Download from", "DOI link", "What to do"]
with open(STATUS_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
    w.writeheader(); w.writerows(results)
print("Saved", STATUS_CSV.resolve())

Saved C:\Users\keg827\Documents\Github_Repos\Galter_Prism_IVDRS_SUDORS_Post_Request\files_status.csv


In [8]:
from pathlib import Path
import shutil

FILES_DIR = Path("files")
BACKUP_DIR = Path("png_backups")

moved = 0
for png in FILES_DIR.rglob("*.png"):
    dest = BACKUP_DIR / png.relative_to(FILES_DIR)
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(png), str(dest))
    print("moved", png, "->", dest)
    moved += 1
print(f"\n{moved} PNG file(s) moved to {BACKUP_DIR.resolve()}")

moved files\ivdrs_reports\IVSU-0001_Child_Abuse_Fatalities_in_Illinois_2015_2022.png -> png_backups\ivdrs_reports\IVSU-0001_Child_Abuse_Fatalities_in_Illinois_2015_2022.png
moved files\ivdrs_reports\IVSU-0002_Veteran_Suicide_Deaths_in_Illinois_2019_2023.png -> png_backups\ivdrs_reports\IVSU-0002_Veteran_Suicide_Deaths_in_Illinois_2019_2023.png
moved files\ivdrs_reports\IVSU-0003_Intimate_Partner_Problem_Related_Suicide_and_Homicide_in_Ill.png -> png_backups\ivdrs_reports\IVSU-0003_Intimate_Partner_Problem_Related_Suicide_and_Homicide_in_Ill.png
moved files\ivdrs_reports\IVSU-0006_Analysis_of_Older_Adult_Suicide_Deaths_in_Illinois_2020_2021.png -> png_backups\ivdrs_reports\IVSU-0006_Analysis_of_Older_Adult_Suicide_Deaths_in_Illinois_2020_2021.png
moved files\ivdrs_reports\IVSU-0007_Child_and_Young_Adult_Suicide_Focus_on_Incidents_with_an_Int.png -> png_backups\ivdrs_reports\IVSU-0007_Child_and_Young_Adult_Suicide_Focus_on_Incidents_with_an_Int.png
moved files\ivdrs_reports\IVSU-0008_Ill